
# 06 · Explainability, Fairness, Robustness (Original Columns)

Original columns only:
- resumes: `resume_id`, `clean_text_redacted` (fallback `clean_text`)
- jds: `job_title`, `jd_text`
- pairs: `row_id`, `job_title`, `label`


In [1]:
RESUMES_CSV = "data/processed/resumes_clean.csv"
JDS_CSV = "data/processed/jd_clean.csv"
PAIRS_CSV = "data/raw/gold_pairs_adzuna.csv"
REPORTS_DIR = "reports"
GROUP_COL = None 

In [47]:
import os, numpy as np, pandas as pd, re, json
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

def must_exist(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"File not found: {path}. Please set the correct path in the first cell.")
    return path

os.makedirs(REPORTS_DIR, exist_ok=True)

resumes = pd.read_csv(must_exist(RESUMES_CSV))
jds     = pd.read_csv(must_exist(JDS_CSV))
pairs   = pd.read_csv(must_exist(PAIRS_CSV))

RESUME_TEXT_COL = "clean_text_redacted" if "clean_text_redacted" in resumes.columns else "clean_text"
assert RESUME_TEXT_COL in resumes.columns, "Expected clean_text_redacted or clean_text in resumes."

In [19]:
# JD text per job title
if "n_chars" in jds.columns:
    jds_agg = jds.sort_values("n_chars", ascending=False).drop_duplicates(["job_title"])
else:
    jds_agg = jds.drop_duplicates(["job_title"])
jd_lookup = dict(zip(jds_agg["job_title"].astype(str), jds_agg["jd_text"].astype(str)))
cat_lookup = dict(zip(jds_agg["job_title"].astype(str), jds_agg.get("category", pd.Series(["All"] * len(jds_agg))).astype(str)))

r_lookup = dict(zip(resumes["row_id"].astype(str), resumes[RESUME_TEXT_COL].astype(str)))
pairs = pairs.dropna(subset=["row_id", "job_title", "label"]).copy()
pairs["row_id"] = pairs["row_id"].astype(str)
pairs["text"] = pairs.apply(lambda r: r_lookup.get(r["row_id"], "") + " [SEP] " + jd_lookup.get(str(r["job_title"]), ""), axis=1)

X_test = pairs["text"].astype(str).tolist()
y = pairs["label"].astype(int).values

vec = TfidfVectorizer(min_df=1)
X = vec.fit_transform(X_test)

X_tr, X_te, y_tr, y_te, pairs_tr, pairs_te = train_test_split(X, y, pairs, test_size=0.3, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=300).fit(X_tr, y_tr)
proba_te = clf.predict_proba(X_te)[:, 1]

### Surrogate Fidelity (R^2)

To check that our **term-level explanations** are faithful to the model's behaviour, a **sparse linear surrogate** (Lasso) on TF-IDF of the same JD-resume pair text is trained to **predict the model's calibrated probability**. If a simple linear combination of terms can closely reproduce the model's outputs, then the per-term contributions displayed are consistent with how the model actually scores examples. 

In [22]:
sur_vec = TfidfVectorizer(min_df=1).fit(pairs_te["text"].astype(str))
X_sur = sur_vec.transform(pairs_te["text"].astype(str))
y_sur = proba_te
sur = Lasso(alpha=1e-3).fit(X_sur, y_sur)
pred_sur = sur.predict(X_sur)

y_mean = np.mean(y_sur)
ss_tot = np.sum((y_sur - y_mean) ** 2) + 1e-8
ss_res = np.sum((y_sur - pred_sur) ** 2)
fidelity_r2 = float(1 - ss_res / ss_tot)

fidelity_r2

0.6960164006414746

#### Counterfactual validity (add one skill token)

**What it tests.**  
A quick sanity check: if we append a **single relevant skill** (e.g., “python”) to the pair text, does the model’s **calibrated probability** go up?

**How it works.**  
We score the original text (`p_before`), then score the same text with the token appended (`p_after`).  
The function returns:  
- `delta = p_after - p_before` and a boolean `changed` (True if `p_after > p_before`).  
If the token already exists, we leave the text unchanged.

**How to read it.**  
- **Positive `delta`** (and `changed=True`) suggests **local monotonicity**: adding a high-weight skill increases confidence, consistent with feature attributions.  
- **Zero/negative `delta`** can happen if the token is neutral/irrelevant in context or outweighed byiting the résumé.


In [29]:
def counterfactual_validity_example(clf, vec, text, feature_to_add="python"):
    X0 = vec.transform([text])
    p0 = clf.predict_proba(X0)[:, 1][0]
    if feature_to_add.lower() in text.lower():
        return {"p_before": float(p0), "p_after": float(p0), "delta": 0.0, "changed": False}

    text2 = text + " " + feature_to_add
    X1 = vec.transform([text2])
    p1 = clf.predict_proba(X1)[:, 1][0]
    return {"p_before": float(p0), "p_after": float(p1), "delta": float(p1 - p0), "changed": bool(p1 > p0)}

example_text = pairs_te.iloc[0]["text"]
cf_result = counterfactual_validity_example(clf, vec, example_text, feature_to_add="python")
cf_result

{'p_before': 0.19967442516326914,
 'p_after': 0.20011196240166199,
 'delta': 0.00043753723839284886,
 'changed': True}

### Robustness: domain shift (by JD category) & keyword-stuffing probe

**Why.**  
We want to know if performance is **stable across JD categories** (domain shift) and whether the model is **vulnerable to keyword stuffing**.

**How.**
1) **Per-category robustness.** We map each pair to a coarse JD category (via `cat_lookup`) and, for each subset, compute:
   - **ROC-AUC** (threshold-free ranking quality),
   - **F1** at a fixed 0.5 cutoff (you can also report F1 at the app’s τ).
   This flags categories where the model underperforms.

2) **Keyword-stuffing stress test.** For a held-out example, we score the original text (`before`) and a copy with a repeated token appended (e.g., `" python"*20` → `after`).  
   We record **Δ = after − before** to see if naive repetition can game the score.

**How to read it.**
- **Per-category ROC-AUC/F1:** similar values across categories ⇒ model is robust to domain; large drops highlight domains needing more data or tuned thresholds.
- **Keyword-stuffing Δ:** ideally **near zero** (no exploit). A large positive Δ indicates vulnerability; consider stronger TF-IDF normalization, stricter stoplists, or clipping used-score design.


In [51]:
pairs_te = pairs_te.copy()
pairs_te["category"] = pairs_te["job_title"].astype(str).map(cat_lookup).fillna("Unknown")

def metric_on_subset(sub):
    if len(sub) == 0:
        return {"roc_auc": None, "f1": None}
    Xs = vec.transform(sub["text"].astype(str).tolist())
    ys = sub["label"].astype(int).values
    ps = clf.predict_proba(Xs)[:, 1]
    pr = (ps >= 0.5).astype(int)
    return {"roc_auc": float(roc_auc_score(ys, ps)), "f1": float(f1_score(ys, pr))}

cats = pairs_te["category"].unique().tolist()
robustness_report = {c: metric_on_subset(pairs_te[pairs_te["category"] == c]) for c in cats[:5]}

def score_text(t):
    return float(clf.predict_proba(vec.transform([t]))[:, 1][0])
before = score_text(example_text)
after = score_text(example_text + " " + (" python" * 20))
robustness_report["adv_keyword"] = {"before": float(before), "after": float(after), "delta": float(after - before)}
robustness_report

{'Hospitality & Catering Jobs': {'roc_auc': 0.8623188405797102,
  'f1': 0.5365853658536586},
 'IT Jobs': {'roc_auc': 0.7024339504888704, 'f1': 0.34355828220858897},
 'Trade & Construction Jobs': {'roc_auc': 0.9058823529411764,
  'f1': 0.7619047619047619},
 'Engineering Jobs': {'roc_auc': 0.8437279774489076, 'f1': 0.4126984126984127},
 'PR, Advertising & Marketing Jobs': {'roc_auc': 0.8533971728226173,
  'f1': 0.6575342465753424},
 'adv_keyword': {'before': 0.19967442516326914,
  'after': 0.20346931823031464,
  'delta': 0.0037948930670455017}}

## Save reports

In [55]:
with open(f"{REPORTS_DIR}/ explainability.json", "w") as f:
    json.dump({"fidelity_r2": fidelity_r2, "counterfactual_example": cf_result}, f, indent=2)
with open (f"{REPORTS_DIR}/ robustness.json", "w") as f:
    json.dump(robustness_report, f, indent=2)
print("Saved to", REPORTS_DIR)

Saved to reports
